# 12 Locale eval
One feature, the help answer, in a second language (Spanish). Build the golden set, run the variant, wire it into CI,
and re-check the judge on Spanish hand labels. Everything is written to `stage-12/`.
Save your run of this notebook as `stage-12/12_locale_eval.ipynb`.

Step 1: set up the kit

In [ ]:
# Setup: in Colab, clone the kit first. Locally, find the kit folder. Change REPO_URL to your fork.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/arpitRanjan/atp-builder-kit"
if "google.colab" in sys.modules and not Path("kit/llm.py").exists():
    if not Path("atp-builder-kit").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("atp-builder-kit")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mcp==2.3.0", "google-genai==3.3.0", "openai==3.51.0"], check=True)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kit" / "llm.py").exists())
sys.path.insert(0, str(ROOT))
CORPUS = Path(os.environ.get("CORPUS_DIR", ROOT / "corpus"))
OUTPUT = Path(os.environ.get("KIT_OUTPUT_DIR", ROOT / "stage-12"))
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Kit folder found:", ROOT.name)

Step 2: the Spanish golden set: input, expected page, source

The kit's 20 Spanish cases (`evals/golden_es.jsonl`) are translations of kit practice questions, plus degraded inputs:
typos and mixed Spanish and English. The notebook copies them to `stage-12/golden-set.jsonl` the first time.
After that the file is yours: edit it or add cases, and the notebook and CI use your version.

In [ ]:
import json, re, logging
import pandas as pd
from kit.llm import call_llm, default_model, pick_provider

for name in ("httpx", "httpx2"):  # the MCP library turns on one log line per HTTP call; hide them
    logging.getLogger(name).setLevel(logging.WARNING)

sys.path.insert(0, str(ROOT / "stage-12"))
sys.path.insert(0, str(ROOT / "evals"))
import locale_evals  # the variant feature and the CI runner: stage-12/locale_evals.py
from gates import hard_failures  # the same hard gates as Chapter 8

GOLDEN = OUTPUT / "golden-set.jsonl"
if not GOLDEN.exists():
    GOLDEN.write_text("".join(json.dumps(c, ensure_ascii=False) + "\n" for c in locale_evals.starter_cases()))
cases = locale_evals.load_cases(GOLDEN)
deg = pd.Series([c["degradation"] for c in cases]).value_counts().to_dict()
print(f"{len(cases)} cases in {GOLDEN.name}:", pd.Series([c["kind"] for c in cases]).value_counts().to_dict(), deg)
assert len(cases) >= 10, "the brief needs at least 10 cases"
assert any("typo" in c["degradation"] for c in cases) and any(c["degradation"] == "mixed_language" for c in cases)
PROVIDER = pick_provider()
print("Model provider for this run:", PROVIDER, default_model(PROVIDER))
pd.DataFrame(cases)[["id", "kind", "degradation", "input", "expected", "source"]]

Step 3: run the app as is on English and Spanish, with the same graders as Stage 8

In [ ]:
from app.app import answer
from mcp_server.server import search

source = {q["qid"]: q for q in map(json.loads, open(CORPUS / "questions.jsonl"))}
starter = {c["description"]: c for c in map(json.loads, open(ROOT / "evals" / "golden_es.jsonl"))}
english = [{**c, "kind": "english", "input": source[starter[c["id"]]["source_qid"]]["question"]}
           for c in cases if c["kind"] == "translated" and c["id"] in starter]

def app_ask(question):
    r = answer(question, folder=CORPUS)
    return {"text": r["text"], "cited": r["cited"]}

base = pd.DataFrame(locale_evals.score(english, app_ask, "as is") + locale_evals.score(cases, app_ask, "as is"))
print("Hard gate pass rate, app as is:")
print(base.groupby("kind", sort=False).passed.agg(["mean", "sum", "count"]).round(2))

Step 4: trace one Spanish failure

In [ ]:
miss = base[(base.kind == "translated") & ~base.passed]
if len(miss):
    row = miss.iloc[0]
    print(f"{row.id}: {row.input}\nWhy it failed: {row.why}\nAnswer: {row.output[:120]}")
    print("Search hits for the Spanish question:", [h["id"] for h in search(row.input, 3, CORPUS)] or "none")
    print("The search matches English words, so a Spanish question finds the wrong pages or none.")
else:
    print("No Spanish failure to trace in this run.")

Step 5: the variant: translate for search, answer in the user's language

The code is in `stage-12/locale_evals.py` (`localized_ask`), so the same function runs here and in CI.
Every output is saved to `stage-12/outputs.jsonl`, and the pass rates to `stage-12/results.csv`.

In [ ]:
fixed = pd.DataFrame(locale_evals.score(cases, lambda q: locale_evals.localized_ask(q, CORPUS), "variant"))
both = pd.concat([base, fixed])
both.to_json(OUTPUT / "outputs.jsonl", orient="records", lines=True, force_ascii=False)
table = both.pivot_table(index="kind", columns="setup", values="passed", aggfunc="mean", sort=False).round(2)
table.to_csv(OUTPUT / "results.csv")
print(table)
print("\nVariant by degradation:", fixed.groupby("degradation").passed.mean().round(2).to_dict())
print("Variant failures:", fixed[~fixed.passed][["id", "why"]].to_dict("records") or "none")

ES_WORDS = {"el", "la", "los", "las", "de", "que", "y", "en", "una", "para", "con", "por", "tu", "tus", "puedes", "nota", "notas"}
def looks_spanish(text):
    return sum(w in ES_WORDS for w in re.findall(r"[a-záéíóúñ]+", text.lower())) >= 3
print("\nAdvisory: answers that look Spanish:", round(fixed.output.map(looks_spanish).mean(), 2),
      "(offline the mock echoes the question, so this number means little; check it live)")

Step 6: the same cases in CI

`stage-12/locale_evals.py` runs the golden set through the variant and exits 1 under the bar. Add it to your
`.github/workflows/evals.yml` as one more step (see `stage-12/README.md`). Its results file is
`stage-12/results/ci-results.json`.

In [ ]:
code_ = locale_evals.main(["--golden", str(GOLDEN), "--out", str(OUTPUT / "results" / "ci-results.json")])
print("Exit code:", code_, "(0 means CI passes)")

Step 7: one judge criterion, re-checked on 10 Spanish hand labels

The criterion: the answer is correct and fully supported by the reference, in Spanish. The exact judge prompt is saved
to `stage-12/judge-prompt.md`. Agreement, with every disagreement, goes to `stage-12/judge-agreement.md`.

In [ ]:
JUDGE_SYSTEM = ("You grade answers from a help assistant. PASS only if the answer is correct and fully "
                "supported by the reference. Any wrong detail, invented feature or refusal is a FAIL. "
                "Reply with PASS or FAIL, a colon, then one short reason.")
JUDGE_USER = "Question: {question}\nReference: {reference}\nAnswer: {answer}\nReply with PASS or FAIL."
(OUTPUT / "judge-prompt.md").write_text(
    "# Judge prompt (Spanish)\n\nCriterion: the answer is correct and fully supported by the reference. "
    "One wrong detail, an invented feature or a refusal is a FAIL.\n\n## System\n\n```\n" + JUDGE_SYSTEM +
    "\n```\n\n## User message\n\n```\n" + JUDGE_USER + "\n```\n\nParse: the first word, PASS or FAIL. "
    "Anything else counts as unparsed.\n")

labels = pd.read_csv(ROOT / "evals" / "labels_es.csv")

def judge(row):
    r = call_llm(JUDGE_USER.format(question=row.question, reference=row.reference, answer=row.answer),
                 system=JUDGE_SYSTEM, max_output_tokens=800)
    verdict = re.match(r"\s*(PASS|FAIL)", r["text"].upper())
    return verdict.group(1).lower() if verdict else "unparsed"

labels["judge_label"] = [judge(row) for row in labels.itertuples()]
labels.to_csv(OUTPUT / "judge-agreement.csv", index=False)
agree = int((labels.human_label == labels.judge_label).sum())
pct = agree / len(labels)
diff = labels[labels.human_label != labels.judge_label]
lines = ["| id | answer | human | judge | note |", "| --- | --- | --- | --- | --- |"]
lines += [f"| {r.id} | {r.answer[:70]} | {r.human_label} | {r.judge_label} | {r.note} |" for r in diff.itertuples()]
(OUTPUT / "judge-agreement.md").write_text(
    f"# Judge agreement in Spanish\n\nProvider `{PROVIDER}`. Agreement: {agree} of {len(labels)} ({pct:.0%}).\n\n"
    "## Where judge and human differed\n\n" + ("\n".join(lines) if len(diff) else "None.") + "\n")
print(pd.crosstab(labels.human_label, labels.judge_label, rownames=["human"], colnames=["judge"]))
print(f"Agreement: {agree} of {len(labels)} ({pct:.0%}). Compare with your English numbers from 08. Disagreements:")
diff[["id", "answer", "human_label", "judge_label", "note"]]